In [ ]:
import requests
import pandas as pd
import json

In [ ]:
url = "https://data.education.gouv.fr/api/explore/v2.1/catalog/datasets/fr-en-college-effectifs-niveau-sexe-lv/records"

params = {    
    "offset": 0
}

response = requests.get(url, params=params)

print(response.status_code)
print(response.json())

In [ ]:
data = response.json()

print(data.keys())
print(data["results"][0])

In [ ]:
df = pd.DataFrame(data["results"])

print(df.head())
print(df.columns.tolist())

In [ ]:
url = "https://data.education.gouv.fr/api/explore/v2.1/catalog/datasets/fr-en-college-effectifs-niveau-sexe-lv/records"

response = requests.get(url, params=params)
data = response.json()

print(data["results"][0].keys())

In [ ]:
url = "https://data.education.gouv.fr/api/explore/v2.1/catalog/datasets/fr-en-college-effectifs-niveau-sexe-lv/records"

params = {
    "where": "code_dept='69'",
    "limit": None
}

response = requests.get(url, params=params)

data = response.json()["results"]

df = pd.DataFrame(data)

print(df[[
    "numero_college",
    "denomination_principale",
    "commune",
    "code_commune",
    "nombre_eleves_total"
]])

In [ ]:
url = "https://geo.api.gouv.fr/departements"

response = requests.get(url)
data = response.json()

df = pd.DataFrame(data)

print(df.head())

In [ ]:
departements = requests.get(
    "https://geo.api.gouv.fr/departements"
).json()

regions = requests.get(
    "https://geo.api.gouv.fr/regions"
).json()

df_dep = pd.DataFrame(departements)
df_reg = pd.DataFrame(regions)

print(df_dep.columns)
print(df_reg.columns)

In [ ]:
url = "https://geo.api.gouv.fr/communes?fields=nom,code,codeDepartement,population&format=json"

data = requests.get(url).json()

df = pd.DataFrame(data)

#print(df.head())
df

In [ ]:
url = "https://api-lannuaire.service-public.gouv.fr/api/explore/v2.1/catalog/datasets/api-lannuaire-administration/records"

params = {
    "limit": 5,
    "where": 'nom like "Mairie%"'
}

response = requests.get(url, params=params)
data = response.json()["results"]

rows = []

for item in data:
    adresse = json.loads(item["adresse"])[0] if item.get("adresse") else {}

    rows.append({
        "code_insee": item.get("code_insee_commune"),
        "nom": item.get("nom"),
        "adresse": adresse.get("numero_voie"),
        "code_postal": adresse.get("code_postal"),
        "commune": adresse.get("nom_commune"),
        "latitude": adresse.get("latitude"),
        "longitude": adresse.get("longitude"),
        "email": item.get("adresse_courriel"),
    })

df = pd.DataFrame(rows)

#print(df.head())
df

In [ ]:
import requests

url = (
    "https://data.education.gouv.fr/api/explore/v2.1/catalog/"
    "datasets/fr-en-lycee_gt-effectifs-niveau-sexe-lv/records"
)

response = requests.get(
    url,
    params={        
        "offset": 0
    }
)

print(response.status_code)

data = response.json()

print(data.keys())

In [ ]:
import pandas as pd

df = pd.DataFrame(data["results"])

print(
    df[
        [
            "numero_lycee",
            "denomination_principale",
            "patronyme",
            "code_commune",
            "commune",
            "nombre_d_eleves",
            "rentree_scolaire",
        ]
    ]
)

In [ ]:
url = "https://www.data.gouv.fr/api/1/datasets/r/98f3161f-79ff-4f16-8f6a-6d571a80fea2"

df = pd.read_csv(
    url,
    sep=";",
    encoding="latin1",
    skiprows=1,
    header=None,
    dtype=str,
    low_memory=False
)

df = df.rename(columns={
    1: "finess",
    3: "nom",
    7: "numero_voie",
    8: "type_voie",
    9: "libelle_voie",
    12: "code_commune_3",
    13: "code_departement",
    15: "code_postal_ville",
    18: "code_categorie",
    19: "libelle_categorie"
})

pharmacies = df[
    df["libelle_categorie"].str.contains("pharm|officine", case=False, na=False)
].copy()

pharmacies["code_insee"] = (
    pharmacies["code_departement"].str.zfill(2)
    + pharmacies["code_commune_3"].str.zfill(3)
)

pharmacies["adresse"] = (
    pharmacies["numero_voie"].fillna("") + " " +
    pharmacies["type_voie"].fillna("") + " " +
    pharmacies["libelle_voie"].fillna("")
).str.strip()

pharmacies_clean = pharmacies[[
    "finess",
    "nom",
    "adresse",
    "code_insee"
]]

#print(pharmacies_clean.head())
pharmacies_clean